# Stage 4B: first GPU baseline, development Fold 0 only
Run cells in order in a Google Colab **GPU** runtime. Setup cells do not train or open images. Only the explicitly labelled training cell starts training. Do not use Run all until settings and preflight have been reviewed.

Push this workflow and its manifest to your Git repository first (raw/processed data stay outside Git). Set a full commit SHA that includes these files. Private repositories require Colab Git authentication; never save tokens in this notebook.

Upload binary copies to Drive, preserving exact case and bytes:
```
MyDrive/faithful-cbm/dataset/
  processed/stage2a/cohort.csv
  raw/release_v0/images/<development derm_path subdirectories and files>
MyDrive/faithful-cbm/outputs/  # separate persistent output root
```
Use the frozen development_ids.csv to select the 658 dermoscopic files using processed cohort.csv's derm_path. Keep locked-test images offline. No clinical images, raw metadata, or original dataset index files are needed. The processed FCL path spelling must match the audited files. Do not rebuild the cohort or splits.


In [ ]:
from pathlib import Path
import json, os, re, subprocess, sys
REPO_URL = "REPLACE_WITH_YOUR_GIT_REPOSITORY_URL"
COMMIT = "REPLACE_WITH_FULL_40_CHARACTER_COMMIT_SHA"
REPO = Path("/content/faithful-medical-cbm")
DATASET = Path("/content/drive/MyDrive/faithful-cbm/dataset")
OUTPUTS = Path("/content/drive/MyDrive/faithful-cbm/outputs")
if not re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT) or REPO_URL.startswith("REPLACE"):
    raise ValueError("Set your repository URL and full pinned commit SHA first")
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer is required")
from google.colab import drive
drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive mount is unavailable")
if not DATASET.is_relative_to(Path("/content/drive/MyDrive")) or not OUTPUTS.is_relative_to(Path("/content/drive/MyDrive")):
    raise ValueError("This workflow requires Drive-backed dataset and outputs")


In [ ]:
def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
if not REPO.exists():
    run(["git", "clone", REPO_URL, str(REPO)])
else:
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("Existing checkout uses a different remote")
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
    if dirty:
        raise RuntimeError("Checkout has changes. Use a fresh Colab runtime; do not overwrite it.")
run(["git", "fetch", "origin", COMMIT], cwd=REPO)
run(["git", "checkout", "--detach", COMMIT], cwd=REPO)
os.chdir(REPO)
print("Pinned source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## Install while preserving the runtime CUDA PyTorch pair
Dependency installation and checks run in fresh subprocesses so the notebook kernel does not retain stale imported packages. A resolver/import failure is a stop condition; do not substitute CPU wheels or change the scientific configuration. Start a fresh compatible GPU runtime if necessary.


In [ ]:
run(["nvidia-smi"])
probe = "import torch, torchvision, json; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(json.dumps({'torch':torch.__version__, 'torchvision':torchvision.__version__, 'gpu':torch.cuda.get_device_name(0)}))"
pair = json.loads(subprocess.check_output([sys.executable, "-c", probe], text=True))
print(pair)
constraints = Path("/content/colab-cuda-constraints.txt")
constraints.write_text("".join(f"{name}=={pair[name]}\n" for name in ("torch", "torchvision")))
run([sys.executable, "-m", "pip", "install", "-c", str(constraints), "-r", "requirements.txt"])
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."])
run([sys.executable, "-c", "import torch, torchvision, timm; assert torch.cuda.is_available(); print('GPU:', torch.cuda.get_device_name(0)); print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'timm', timm.__version__, 'CUDA', torch.version.cuda, 'cuDNN', torch.backends.cudnn.version())"])


## Connect storage and validate frozen inputs
This verifies the local Stage 4 snapshot of configuration and sources, exact frozen split/cohort/summary hashes, all 658 development path existences and Fold 0 loader membership. It never iterates a loader or opens an image. Historical config hashes in Stage 2 artifacts describe earlier-stage configurations; the accompanying manifest records the approved current baseline config. Only text source/config CRLF is normalized for cross-platform hashing; frozen data artifacts are byte-exact.

Data roots are symlinked as whole directories so existing loader containment checks remain valid. Output parent directories are symlinked to Drive; final run directories are left for the trainer to create. A preexisting run refuses training, including after interruption. Exact interrupted-run resume is not implemented.


In [ ]:
sys.path.insert(0, str(REPO / "docs"))
from colab_fold0 import wire_storage, preflight, summarize
wire_storage(REPO, DATASET, OUTPUTS)
report = preflight(REPO, OUTPUTS)
print(json.dumps(report, indent=2))
SETUP = OUTPUTS / "setup/baseline-v1"
SETUP.mkdir(parents=True, exist_ok=True)
(SETUP / "preflight.json").write_text(json.dumps(report, indent=2))
(SETUP / "default.toml").write_bytes((REPO / "configs/default.toml").read_bytes())
(SETUP / "frozen_manifest.json").write_bytes((REPO / "docs/colab_fold0_manifest.json").read_bytes())
environment = subprocess.check_output([sys.executable, "scripts/report_environment.py"], text=True)
(SETUP / "environment.json").write_text(environment)
(SETUP / "pip-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print(environment)


## Start the real GPU run — Fold 0 ONLY
Execute this cell manually after successful preflight. It rechecks inputs immediately before launch. The first actual model construction downloads ImageNet weights if uncached. Internet/weight-cache access and sufficient Drive space are required.

Exact command:
```
python -m faithful_medical_cbm.training.train_black_box --config configs/default.toml --fold 0 --run-name baseline-v1
```
Training uses 493 development cases (148 positive/345 negative); validation uses 165 (50/115). No locked-test loader is constructed. Case-level independence only: patient-level independence cannot be verified.


In [ ]:
preflight(REPO, OUTPUTS)
run([sys.executable, "-m", "faithful_medical_cbm.training.train_black_box",
     "--config", "configs/default.toml", "--fold", "0", "--run-name", "baseline-v1"], cwd=REPO)


## Development-only post-run summary
Persistent outputs:
- `outputs/checkpoints/baseline/baseline-v1/fold_0/{best,last}.pt`
- `outputs/artifacts/baseline/baseline-v1/fold_0/`: raw `validation_epoch_NNN.csv`, `history.csv`, `run.json` (config and provenance), `summary.json`
- `outputs/setup/baseline-v1/`: environment, package freeze, exact config, manifest, preflight

The following cell reads reports and checks checkpoint existence without loading weights. On interruption it reports an incomplete run; preserve partial artifacts and decide recovery separately. Do not delete outputs or automatically restart. Do not continue to other folds or test evaluation.


In [ ]:
result = summarize(OUTPUTS)
print(json.dumps(result, indent=2))
(SETUP / "post_run_summary.json").write_text(json.dumps(result, indent=2))
